### （1)加载环境变量

In [1]:
import os

api_key=os.getenv('AGNES_API_KEY')
# print(api_key)


### 文生图

In [2]:
import requests
import time
import json
import os

# Load API key from environment variable
api_key = os.getenv('AGNES_API_KEY')
if not api_key:
    raise ValueError("请先设置 AGNES_API_KEY 环境变量")

# Step 1: 创建视频生成任务
url = "https://apihub.agnes-ai.com/v1/videos"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
payload = {
    "model": "agnes-video-v2.0",
    "prompt": "A cinematic shot of a cat walking on the beach at sunset, soft ocean waves, warm golden lighting, realistic motion",
    "height": 768,
    "width": 1152,
    "num_frames": 121,
    "frame_rate": 24
}

print("🚀 正在创建视频生成任务...")
response = requests.post(url, headers=headers, json=payload)
response.raise_for_status()
task_data = response.json()
print(f"✅ 任务创建成功: {json.dumps(task_data, indent=2, ensure_ascii=False)}")

video_id = task_data.get("video_id") or task_data.get("task_id") or task_data.get("id")
print(f"\n📌 Video ID: {video_id}")

# Step 2: 轮询任务状态直到完成
status_url = f"https://apihub.agnes-ai.com/agnesapi?video_id={video_id}"
status_headers = {"Authorization": f"Bearer {api_key}"}

print("\n⏳ 等待视频生成完成...")
while True:
    status_response = requests.get(status_url, headers=status_headers)
    status_response.raise_for_status()
    status_data = status_response.json()

    current_status = status_data.get("status")
    progress = status_data.get("progress", 0)
    seconds = status_data.get("seconds", 0)
    size = status_data.get("size", 0)
    print(f"   状态: {current_status}, 进度: {progress}% , 视频时长，单位为秒：{seconds}, 	视频分辨率:{size}")

    if current_status == "completed":
        print("\n✅ 视频生成完成！")
        video_url = status_data.get("metadata", {}).get("url")
        if video_url:
            print(f"🎬 视频 URL: {video_url}")
        else:
            print("⚠️ 响应中未找到视频 URL")
            print(f"完整响应: {json.dumps(status_data, indent=2, ensure_ascii=False)}")
        break
    elif current_status in ("failed", "error"):
        print(f"❌ 视频生成失败！")
        print(f"完整响应: {json.dumps(status_data, indent=2, ensure_ascii=False)}")
        break

    time.sleep(5)


🚀 正在创建视频生成任务...
✅ 任务创建成功: {
  "id": "task_pkjvGF2Bg9AIDewzSubSNVRJGpqjyFI2",
  "video_id": "task_pkjvGF2Bg9AIDewzSubSNVRJGpqjyFI2",
  "task_id": "task_pkjvGF2Bg9AIDewzSubSNVRJGpqjyFI2",
  "object": "video",
  "model": "agnes-video-v2.0",
  "status": "queued",
  "progress": 0,
  "created_at": 1784635623,
  "seconds": "5.0",
  "size": "1152x768"
}

📌 Video ID: task_pkjvGF2Bg9AIDewzSubSNVRJGpqjyFI2

⏳ 等待视频生成完成...
   状态: in_progress, 进度: 30% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832
   状态: in_progress, 进度: 30% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832
   状态: in_progress, 进度: 30% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832
   状态: in_progress, 进度: 30% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832
   状态: in_progress, 进度: 30% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832
   状态: in_progress, 进度: 30% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832
   状态: in_progress, 进度: 30% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832
   状态: in_progress, 进度: 30% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832
   状态: completed, 进度: 100% , 视频时长，单位为秒：5.0, 	视频分辨率:1088x832

✅ 视频生成完成！
⚠️ 响应中未找到视频 URL
完整响应: {
  "

### 图生视频

In [2]:
import requests
import time
import json
import os

# 图生视频 - 使用前面生成的图片作为输入
api_key = os.getenv('AGNES_API_KEY')
if not api_key:
    raise ValueError("请先设置 AGNES_API_KEY 环境变量")

# 图片 URL（来自前面图生图步骤的输出）
image_url = "https://platform-outputs.agnes-ai.space/images/i2i/task_8yPE915Fce7pDnIhCP8PoBwaw8U5JlTB/output.png"

# 创建视频生成任务
url = "https://apihub.agnes-ai.com/v1/videos"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# 优化后的 prompt：图中老鹰在飞，悬浮城池，能量圈在波动
prompt = (
    "A majestic eagle soaring gracefully above a floating celestial city suspended in the clouds, "
    "ancient mystical architecture with glowing energy rings pulsing with ethereal blue light, "
    "cinematic aerial shot, slow camera movement, dramatic lighting, "
    "volumetric clouds, epic fantasy landscape, 8k quality, highly detailed"
)

payload = {
    "model": "agnes-video-v2.0",
    "prompt": prompt,
    "image": image_url,
    "num_frames": 121,
    "frame_rate": 24
}

print("🚀 正在创建图生视频任务...")
print(f"图片 URL: {image_url}")
print(f"Prompt: {prompt}")
response = requests.post(url, headers=headers, json=payload)
response.raise_for_status()
task_data = response.json()
print(f"✅ 任务创建成功: {json.dumps(task_data, indent=2, ensure_ascii=False)}")

video_id = task_data.get("video_id") or task_data.get("task_id") or task_data.get("id")

# Step 2: 轮询任务状态直到完成
if not video_id:
    print("❌ 没有 video_id，无法轮询任务状态")
else:
    print(f"\n📌 Video ID: {video_id}")
    status_headers = {"Authorization": f"Bearer {api_key}"}
    print("\n⏳ 等待视频生成完成...")
    while True:
        status_url = f"https://apihub.agnes-ai.com/agnesapi?video_id={video_id}"
        status_response = requests.get(status_url, headers=status_headers)
        status_response.raise_for_status()
        status_data = status_response.json()

        current_status = status_data.get("status")
        progress = status_data.get("progress", 0)
        seconds = status_data.get("seconds", 0)
        size = status_data.get("size", 0)
        print(f"   状态: {current_status}, 进度: {progress}% , 视频时长（秒）: {seconds}, 分辨率: {size}")

        if current_status == "completed":
            print("\n✅ 视频生成完成！")
            video_url = status_data.get("metadata", {}).get("url")
            if video_url:
                print(f"🎬 视频 URL: {video_url}")
            else:
                print("⚠️ 响应中未找到视频 URL")
                print(f"完整响应: {json.dumps(status_data, indent=2, ensure_ascii=False)}")
            break
        elif current_status in ("failed", "error"):
            print(f"❌ 视频生成失败！")
            print(f"完整响应: {json.dumps(status_data, indent=2, ensure_ascii=False)}")
            break

        time.sleep(5)



🚀 正在创建图生视频任务...
图片 URL: https://platform-outputs.agnes-ai.space/images/i2i/task_8yPE915Fce7pDnIhCP8PoBwaw8U5JlTB/output.png
Prompt: A majestic eagle soaring gracefully above a floating celestial city suspended in the clouds, ancient mystical architecture with glowing energy rings pulsing with ethereal blue light, cinematic aerial shot, slow camera movement, dramatic lighting, volumetric clouds, epic fantasy landscape, 8k quality, highly detailed
✅ 任务创建成功: {
  "id": "task_iry4ykUnyTX77KU1nqinPwigFZQs3KQR",
  "video_id": "task_iry4ykUnyTX77KU1nqinPwigFZQs3KQR",
  "task_id": "task_iry4ykUnyTX77KU1nqinPwigFZQs3KQR",
  "object": "video",
  "model": "agnes-video-v2.0",
  "status": "queued",
  "progress": 0,
  "created_at": 1784639172,
  "seconds": "5.0"
}

📌 Video ID: task_iry4ykUnyTX77KU1nqinPwigFZQs3KQR

⏳ 等待视频生成完成...
   状态: queued, 进度: 0% , 视频时长（秒）: 5.0, 分辨率: 1088x832
   状态: queued, 进度: 0% , 视频时长（秒）: 5.0, 分辨率: 1088x832
   状态: queued, 进度: 0% , 视频时长（秒）: 5.0, 分辨率: 1088x832
   状态: queued, 进度: